# TP 1 : reconnaître les maladies et les acariens des feuilles d'olivier

**Objectif.** Construire un classifieur d'images qui distingue trois états d'une
feuille d'olivier :

| Classe | Cause | Ce qu'on voit |
|---|---|---|
| `sain` | — | feuille verte, uniforme |
| `oeil_de_paon` | champignon *Venturia oleaginea* | taches rondes brunes cerclées de jaune |
| `acariose` | acarien *Aculus olearius* (un parasite) | feuille déformée, décolorée, gaufrée |

Puis vérifier honnêtement ce que vaut ce modèle : doublons, fuites entre
entraînement et test, incertitude, et surtout **images venues d'ailleurs**.

**Ce que vous allez apprendre.**

1. Explorer un jeu d'images et trouver ses pièges (doublons, fonds, déséquilibre).
2. Découper train / validation / test sans fuite.
3. Poser une référence simple (histogrammes de couleur + régression logistique).
4. Entraîner un réseau pré-entraîné (*transfer learning*) avec PyTorch et timm.
5. Évaluer : matrice de confusion, rappel par classe, seuil « incertain ».
6. Tester sur deux jeux d'autres pays (Turquie, Pérou) : le vrai test.
7. Voir où regarde le réseau (Grad-CAM), sauvegarder et réutiliser le modèle.

**Comment travailler.** Les cellules marquées « À FAIRE » sont à compléter. Les
questions ❓ sont à traiter dans votre compte rendu. Le corrigé est dans
`tp/corriges/`. Ne l'ouvrez qu'après avoir essayé.

**Données** (à télécharger une fois, depuis la racine du projet, dans un terminal) :

```bash
uv run agridrone data download olive_leaf_uguz_uysal olive_leaf_peacock_diker olive_leaf_peacock_peru
```

## 0. Mise en route

In [ ]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch

# Racine du projet : le dossier qui contient pyproject.toml (le notebook peut être lancé
# depuis tp/ ou tp/corriges/).
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data" / "interim"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
# Sous Windows et macOS, les DataLoader à plusieurs processus ne fonctionnent pas avec des
# classes définies dans un notebook : on charge les images dans le processus principal.
NUM_WORKERS = 2 if sys.platform.startswith("linux") else 0
print(f"Projet : {ROOT}\nMatériel : {DEVICE}")

import hashlib
import time

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

OUT = ROOT / "runs" / "tp1"
OUT.mkdir(parents=True, exist_ok=True)

# Réglages : sur CPU, un entraînement court (quelques minutes par époque) ; sur GPU, complet.
MODE_RAPIDE = DEVICE == "cpu"
IMG_SIZE = 192 if MODE_RAPIDE else 224
EPOCHS = 4 if MODE_RAPIDE else 15
BATCH = 32
ARCH = "efficientnet_b0.ra_in1k"   # 5,3 M de paramètres, poids ImageNet
CLASSES = ["sain", "oeil_de_paon", "acariose"]
print(f"MODE_RAPIDE={MODE_RAPIDE}, IMG_SIZE={IMG_SIZE}, EPOCHS={EPOCHS}")

## 1. Les données

Le jeu d'Uğuz et Uysal (Denizli, Turquie, 2019) contient 3 400 photos de feuilles en
cinq archives. Les auteurs fournissent un découpage « officiel » : des archives
d'entraînement et une archive de test. Deux dossiers ne portent pas de nom de classe
(`train3`, `train4`) : ce sont des feuilles atteintes d'œil de paon (vérifié à l'œil).

In [ ]:
UGUZ = DATA / "olive_leaf_uguz_uysal"
DOSSIERS = {  # dossier -> (classe, découpage officiel)
    "Train1/Healthy": ("sain", "train"),
    "Train2/aculus_olearius": ("acariose", "train"),
    "train3/train3": ("oeil_de_paon", "train"),
    "train4/train4": ("oeil_de_paon", "train"),
    "test/test/Healthy": ("sain", "test"),
    "test/test/aculus_olearius": ("acariose", "test"),
    "test/test/olive_peacock_spot": ("oeil_de_paon", "test"),
}
EXTENSIONS = {".jpg", ".jpeg", ".png"}

lignes = []
for dossier, (classe, officiel) in DOSSIERS.items():
    for f in sorted((UGUZ / dossier).iterdir()):
        if f.suffix.lower() in EXTENSIONS:
            lignes.append({"chemin": f, "classe": classe, "officiel": officiel})
df = pd.DataFrame(lignes)
print(len(df), "images")
df.head()

### Exercice 1.1 : combien d'images par classe ?

Affichez un tableau croisé classe × découpage officiel (`pd.crosstab`) et un
diagramme en barres. Les classes sont-elles équilibrées ?

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
# tableau = pd.crosstab(...)
# tableau.plot.bar(...)

### Exercice 1.2 : regarder les images

Écrivez `montrer(df, n)`, qui affiche `n` images tirées au hasard pour chaque classe.

❓ Quels indices visuels distinguent les classes ? Sur quel fond sont prises les
photos ? Pourquoi est-ce un risque pour un modèle destiné à des images prises au
champ ou par drone ?

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
def montrer(df, n=5, graine=0):
    ...  # une ligne par classe, n images par ligne (plt.subplots, ax.imshow, ax.axis("off"))

montrer(df)

### Exercice 1.3 : les doublons exacts

Une même photo présente deux fois peut se retrouver à la fois en entraînement et en
test. Le modèle est alors évalué sur des images qu'il a apprises : c'est une
**fuite**, qui gonfle les scores.

Calculez l'empreinte MD5 de chaque fichier (`hashlib.md5(chemin.read_bytes())`),
puis comptez :

- les fichiers en double ;
- les photos présentes **à la fois** dans le train et le test officiels ;
- les photos rangées dans deux classes différentes.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
# df["md5"] = ...
# nb_doublons = ...
# fuites = ...      # empreintes présentes dans les deux découpages officiels
# conflits = ...    # empreintes associées à deux classes

### Exercice 1.4 : les quasi-doublons

Une photo recadrée, recompressée ou reprise une seconde plus tard n'a pas la même
empreinte MD5, mais c'est presque la même image. On les repère avec une **empreinte
perceptuelle** (*dHash*) :

1. passer l'image en niveaux de gris, réduite à 9 × 8 pixels ;
2. comparer chaque pixel à son voisin de droite, ce qui donne 64 bits ;
3. deux images sont quasi identiques si moins de quelques bits diffèrent (distance de
   Hamming).

Complétez `dhash`, calculez la matrice des distances (fournie), puis regroupez les
images proches : deux images à distance ≤ `SEUIL` sont dans le même **groupe**. Ces
groupes serviront au découpage : un groupe entier va soit en entraînement, soit en
validation.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
def dhash(chemin, taille=8):
    """Empreinte perceptuelle sur 64 bits (entier Python)."""
    ...

df["dhash"] = [dhash(p) for p in df["chemin"]]

# Fourni : distances de Hamming entre toutes les paires (numpy >= 2.0).
h = np.array(df["dhash"].tolist(), dtype=np.uint64)
distances = np.bitwise_count(h[:, None] ^ h[None, :])
SEUIL = 4

# Regroupement (union-find) : fourni.
parent = list(range(len(df)))
def racine(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i, j in zip(*np.nonzero(np.triu(distances <= SEUIL, k=1))):
    parent[racine(i)] = racine(j)
df["groupe"] = [racine(i) for i in range(len(df))]

❓ Combien de groupes sont à cheval sur le train et le test officiels ? Que
faudrait-il faire si l'on voulait un test parfaitement propre ?

## 2. Découper sans fuite

Règles retenues :

- **test** = le test officiel, **moins** les images dont un quasi-doublon est en
  entraînement (on ne garde que des images jamais vues) ;
- **validation** = 20 % des groupes du train officiel, tirés par classe
  (`StratifiedGroupKFold`) ; elle sert à choisir l'époque et les seuils ;
- **entraînement** = le reste, sans doublon exact.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
from sklearn.model_selection import StratifiedGroupKFold

# 1. groupes présents dans le train officiel
# 2. test = test officiel dont le groupe n'est pas dans le train officiel
# 3. train officiel dédoublonné (drop_duplicates sur "md5"), puis StratifiedGroupKFold(n_splits=5)
#    -> une partie pour la validation, le reste pour l'entraînement
# train, val, test = ...

## 3. Une référence sans deep learning

Avant un réseau de neurones, posez une **référence** simple : si elle fait déjà 95 %,
le problème est facile, ou biaisé. Ici, chaque image est résumée par l'histogramme de
ses couleurs (teinte, saturation, valeur), classé par une régression logistique.

### Exercice 3.1

Complétez `histo_hsv`, entraînez la régression sur `train`, puis évaluez-la sur `val`
(exactitude et rappel par classe).

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def histo_hsv(chemin, bins=16):
    """Concatène les histogrammes normalisés des 3 canaux HSV (3 x bins valeurs)."""
    ...

# X_train = np.stack([...]) ; X_val = ...
# reference = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
# reference.fit(...) ; print(classification_report(...))

❓ Quelle classe la couleur seule reconnaît-elle le mieux ? Pourquoi l'acariose
est-elle plus difficile ?

## 4. Un réseau pré-entraîné (*transfer learning*)

On part d'EfficientNet-B0, entraîné sur ImageNet (1,2 million de photos) : ses
premières couches savent déjà détecter bords, textures et formes. On remplace sa
dernière couche par une couche à 3 sorties, puis on ré-entraîne tout le réseau sur
nos feuilles.

### 4.1 Le jeu de données PyTorch et les augmentations (fourni)

Les augmentations (recadrage, rotations, retournements, couleur) montrent au réseau des
variantes de chaque photo : il apprend des motifs plutôt que des pixels précis.

In [ ]:
import timm
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

MOYENNE, ECART = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # normalisation ImageNet
tf_train = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(0.2, 0.2, 0.2, 0.02),
    T.ToTensor(),
    T.Normalize(MOYENNE, ECART),
])
tf_eval = T.Compose([
    T.Resize(int(IMG_SIZE * 1.15)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(MOYENNE, ECART),
])

class Feuilles(Dataset):
    def __init__(self, chemins, etiquettes, transform):
        self.chemins, self.etiquettes, self.transform = list(chemins), list(etiquettes), transform

    def __len__(self):
        return len(self.chemins)

    def __getitem__(self, i):
        im = Image.open(self.chemins[i])
        im.draft("RGB", (2 * IMG_SIZE, 2 * IMG_SIZE))  # JPEG : décodage réduit, plus rapide
        return self.transform(im.convert("RGB")), self.etiquettes[i]

def chargeur(d, transform, melanger=False):
    y = [CLASSES.index(c) for c in d["classe"]]
    return DataLoader(Feuilles(d["chemin"], y, transform), batch_size=BATCH, shuffle=melanger,
                      num_workers=NUM_WORKERS)

dl_train = chargeur(train, tf_train, melanger=True)
dl_val, dl_test = chargeur(val, tf_eval), chargeur(test, tf_eval)
x, y = next(iter(dl_train))
print(x.shape, y[:10])

### Exercice 4.2 : la boucle d'entraînement

C'est le cœur de tout entraînement PyTorch. Écrivez :

- `entrainer_une_epoque(modele, chargeur, optimiseur, critere)` : pour chaque lot,
  remettre les gradients à zéro, calculer la perte, rétropropager, faire un pas
  d'optimisation. Renvoyer la perte moyenne et l'exactitude ;
- `predire(modele, chargeur)` : en mode évaluation et sans gradient, renvoyer les
  probabilités (softmax) et les vraies étiquettes.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
def entrainer_une_epoque(modele, chargeur, optimiseur, critere):
    modele.train()
    ...
    return perte_moyenne, exactitude

@torch.no_grad()
def predire(modele, chargeur):
    modele.eval()
    ...
    return probas, etiquettes   # tableaux numpy (n, 3) et (n,)

### 4.3 Entraîner

- **Poids de classe** : la perte pèse davantage les classes rares.
- **Lissage des étiquettes** (0,1) : le réseau devient moins sûr de lui, ce qui rend ses
  probabilités plus fiables.
- **Taux d'apprentissage** en cosinus : il décroît doucement jusqu'à la fin.
- **On garde la meilleure époque** sur la validation (pas la dernière).

Durée indicative : environ 1 à 2 minutes par époque sur GPU ; sur CPU, comptez 5 à
10 minutes par époque en `MODE_RAPIDE`.

In [ ]:
import copy

from sklearn.utils.class_weight import compute_class_weight

modele = timm.create_model(ARCH, pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
poids = compute_class_weight("balanced", classes=np.arange(len(CLASSES)),
                             y=[CLASSES.index(c) for c in train["classe"]])
critere = nn.CrossEntropyLoss(weight=torch.tensor(poids, dtype=torch.float32, device=DEVICE),
                              label_smoothing=0.1)
optimiseur = torch.optim.AdamW(modele.parameters(), lr=3e-4, weight_decay=0.05)
planning = torch.optim.lr_scheduler.CosineAnnealingLR(optimiseur, T_max=EPOCHS)

historique, meilleur, meilleur_etat = [], -1.0, None
for epoque in range(1, EPOCHS + 1):
    t0 = time.time()
    perte, exact_train = entrainer_une_epoque(modele, dl_train, optimiseur, critere)
    planning.step()
    p_val, y_val = predire(modele, dl_val)
    exact_val = (p_val.argmax(1) == y_val).mean()
    historique.append({"epoque": epoque, "perte": perte, "train": exact_train, "val": exact_val})
    if exact_val > meilleur:
        meilleur, meilleur_etat = exact_val, copy.deepcopy(modele.state_dict())
    print(f"époque {epoque} : perte {perte:.3f}, train {exact_train:.3f}, val {exact_val:.3f} "
          f"({time.time() - t0:.0f} s)")

modele.load_state_dict(meilleur_etat)
pd.DataFrame(historique).set_index("epoque")[["train", "val"]].plot(figsize=(5, 3), marker="o")
plt.ylabel("exactitude"); plt.show()

❓ L'exactitude d'entraînement est-elle plus haute ou plus basse que celle de
validation ? Pourquoi les augmentations peuvent-elles inverser l'écart habituel ?

## 5. Évaluer correctement

### Exercice 5.1 : sur le test

Calculez les probabilités sur `dl_test`, affichez le rapport de classification
(précision, rappel, F1 par classe) et la matrice de confusion
(`ConfusionMatrixDisplay`).

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
from sklearn.metrics import ConfusionMatrixDisplay

# p_test, y_test = predire(...)
# print(classification_report(...))
# ConfusionMatrixDisplay.from_predictions(...)

❓ Pour un oléiculteur, quelle erreur coûte le plus cher : une feuille malade déclarée
saine, ou une feuille saine déclarée malade ? Lisez-la dans la matrice.

### Exercice 5.2 : savoir dire « je ne sais pas »

Le modèle renvoie une probabilité par classe. Quand la plus haute est faible, mieux
vaut répondre **« incertain »** et faire vérifier la feuille.

1. Sur la **validation**, tracez l'exactitude en fonction de la part d'images gardées
   (on garde les plus sûres d'abord).
2. Choisissez le seuil de confiance le plus bas qui garde 99 % d'exactitude sur la
   validation.
3. Appliquez ce seuil au **test** : quelle part d'images est décidée, avec quelle
   exactitude ?

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
def courbe_risque_couverture(probas, y):
    """Renvoie (couverture, exactitude, seuil), triés par confiance décroissante."""
    ...

# couverture, exactitude, seuils = courbe_risque_couverture(p_val, y_val)
# SEUIL_INCERTAIN = ...

## 6. Le vrai test : des feuilles d'autres pays

Le test précédent vient des **mêmes** photographes, du même appareil et de la même
région que l'entraînement. Deux autres jeux, jamais vus, mesurent la
**généralisation** :

- **Turquie** (Diker et al. 2023) : `sağlam` (sain) et `hastalıklı` (œil de paon) ;
- **Pérou** (vallée d'Algarrobal, 2024) : `Healthy` et `Peacock_Spot`, sur fond blanc.

Ils ne contiennent pas d'acariose. On mesure donc surtout « sain contre malade ».

### Exercice 6.1

Construisez un tableau `externe` (chemin, classe, source) pour les deux jeux, prédisez,
puis calculez pour chaque source :

- l'exactitude sain / malade, en comptant comme « malade » toute prédiction autre que
  `sain` ;
- le rappel des malades (part des feuilles malades détectées) ;
- les fausses alertes (part des feuilles saines déclarées malades).

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
SOURCES = {
    "Turquie (Diker)": (DATA / "olive_leaf_peacock_diker" / "zeytin", {"sağlam": "sain", "hastalıklı": "oeil_de_paon"}),
    "Pérou": (DATA / "olive_leaf_peacock_peru" / "peacock-spot-detection-in-olive-leaves" / "Olive_PNG_Dataset",
              {"Healthy": "sain", "Peacock_Spot": "oeil_de_paon"}),
}
# externe = pd.DataFrame(...)
# pour chaque source : predire(...), puis exactitude sain/malade, rappel malade, fausses alertes

❓ Comparez avec le test de la partie 5. D'où peut venir l'écart : appareil photo,
fond, variété d'olivier, stade de la maladie, éclairage ? Qu'en concluez-vous pour des
images prises par drone dans une oliveraie française ?

## 7. Où regarde le réseau ? (Grad-CAM)

Grad-CAM colore les zones de l'image qui ont le plus pesé dans la décision. Si le
réseau regarde le fond plutôt que les taches, il a appris un raccourci.

### Exercice 7.1

Complétez la classe `GradCAM`. Un *hook* mémorise la sortie de la dernière couche de
convolution (`modele.conv_head`) et son gradient. La carte est la somme des cartes
d'activation, pondérées par la moyenne de leur gradient, passée dans un ReLU, puis
normalisée entre 0 et 1.

In [ ]:
# À FAIRE : complétez cette cellule, puis exécutez-la.
class GradCAM:
    def __init__(self, modele, couche):
        self.modele, self.activation, self.gradient = modele, None, None
        def memoriser(module, entree, sortie):
            if sortie.requires_grad:  # rien à mémoriser pendant predire() (sans gradient)
                self.activation = sortie
                sortie.register_hook(lambda g: setattr(self, "gradient", g))
        couche.register_forward_hook(memoriser)

    def __call__(self, x, classe=None):
        """x : tenseur (1, 3, H, W). Renvoie (carte numpy h x w dans [0, 1], classe)."""
        ...

cam = GradCAM(modele, modele.conv_head)

In [ ]:
# Fourni : affiche l'image, puis la carte Grad-CAM, pour quelques erreurs et réussites du test.
def voir_gradcam(lignes, titre):
    fig, axes = plt.subplots(2, len(lignes), figsize=(2.8 * len(lignes), 5.2))
    for k, (_, r) in enumerate(lignes.iterrows()):
        im = Image.open(r["chemin"]).convert("RGB")
        x = tf_eval(im).unsqueeze(0)
        carte, c = cam(x)
        vue = np.asarray(T.CenterCrop(IMG_SIZE)(T.Resize(int(IMG_SIZE * 1.15))(im)))
        axes[0, k].imshow(vue); axes[0, k].set_title(f"vrai : {r['classe']}", fontsize=8)
        axes[1, k].imshow(vue); axes[1, k].imshow(np.asarray(Image.fromarray(np.uint8(carte * 255)).resize(vue.shape[1::-1], Image.BILINEAR)), alpha=0.5, cmap="jet")
        axes[1, k].set_title(f"prédit : {CLASSES[c]}", fontsize=8)
        for ax in axes[:, k]:
            ax.axis("off")
    fig.suptitle(titre); plt.tight_layout(); plt.show()

test_pred = test.assign(predit=[CLASSES[i] for i in p_test.argmax(1)])
erreurs = test_pred[test_pred["classe"] != test_pred["predit"]]
voir_gradcam(erreurs.head(5) if len(erreurs) else test_pred.head(5), "Erreurs du test")
voir_gradcam(test_pred[test_pred["classe"] == test_pred["predit"]].sample(5, random_state=0), "Réussites du test")

❓ Le réseau regarde-t-il les taches et les déformations, ou les bords et le fond ?

## 8. Sauvegarder et réutiliser le modèle

In [ ]:
chemin_modele = OUT / "modele_feuilles.pt"
torch.save({"arch": ARCH, "classes": CLASSES, "img_size": IMG_SIZE,
            "seuil_incertain": SEUIL_INCERTAIN, "state_dict": modele.state_dict()}, chemin_modele)

def diagnostiquer(chemin_image, chemin_modele=chemin_modele):
    """Charge le modèle sauvegardé et renvoie (classe ou 'incertain', probabilités)."""
    ck = torch.load(chemin_modele, map_location="cpu", weights_only=True)
    m = timm.create_model(ck["arch"], pretrained=False, num_classes=len(ck["classes"]))
    m.load_state_dict(ck["state_dict"]); m.eval()
    x = tf_eval(Image.open(chemin_image).convert("RGB")).unsqueeze(0)
    with torch.no_grad():
        p = torch.softmax(m(x), dim=1)[0].numpy()
    reponse = ck["classes"][p.argmax()] if p.max() >= ck["seuil_incertain"] else "incertain"
    return reponse, dict(zip(ck["classes"], p.round(3)))

print(chemin_modele)
diagnostiquer(test["chemin"].iloc[0])

## 9. Pour aller plus loin

1. **Modèle binaire sain / malade** : réentraînez avec deux classes, puis comparez le
   rappel des malades sur les jeux du Pérou et de Turquie.
2. **Adapter au nouveau domaine** : ajoutez 20 images du Pérou à l'entraînement, et
   testez sur le reste. Combien d'images suffisent pour regagner l'exactitude perdue ?
3. **Un autre réseau** : `convnext_tiny.fb_in22k_ft_in1k` (plus gros, souvent plus
   robuste hors domaine). Le gain vaut-il le temps de calcul ?
4. **Le projet complet** : la même démarche, outillée et reproductible (configs,
   MLflow, export ONNX), est dans `uv run agridrone train -c configs/classification/...`
   (voir le README et `reports/baseline_results.md` pour le blé).

## Compte rendu attendu

Une à deux pages :
- le tableau des effectifs ;
- les doublons et fuites trouvés ;
- la référence contre le réseau ;
- la matrice de confusion du test ;
- le seuil « incertain » ;
- le tableau hors domaine ;
- deux cartes Grad-CAM commentées ;
- vos réponses aux ❓.